In [0]:
%sql
CREATE OR REPLACE TEMP VIEW product_category_name_translation_typed AS
SELECT
  product_category_name
  , product_category_name_english
  , _ingested_at
FROM workspace.bronze.product_category_name_translation
QUALIFY ROW_NUMBER() OVER (PARTITION BY product_category_name ORDER BY _ingested_at DESC) = 1;

CREATE OR REPLACE TEMP VIEW product_category_name_translation_validated AS
SELECT *,
  CASE
    WHEN product_category_name IS NULL THEN 'llave nula'
    WHEN product_category_name_english IS NULL THEN 'traduccion no encontrada'
  END AS motivo
FROM product_category_name_translation_typed;

CREATE OR REPLACE TABLE workspace.silver.product_category_name_translation
AS
SELECT * EXCEPT(motivo) FROM product_category_name_translation_validated
WHERE motivo IS NULL;

CREATE OR REPLACE TABLE workspace.silver.product_category_name_translation_quarantine
AS
SELECT * FROM product_category_name_translation_validated
WHERE motivo IS NOT NULL;

In [0]:

%sql
Select 'bronze' as layer, count(*) from workspace.bronze.product_category_name_translation
UNION ALL
Select 'silver' as layer, count(*) from workspace.silver.product_category_name_translation
UNION all
Select 'quarantine' as layer, count(*) from workspace.silver.product_category_name_translation_quarantine